# 02 – Explore item metadata (meta_Video_Games)
Goal: check whether item metadata covers all 25,612 items, especially the 1,542 unseen in train+valid, how complete each field is, and what kinds of items the catalog contains.

## 1. Coverage: does every item have a metadata row?

In [1]:
from pathlib import Path
import pandas as pd

# pd.set_option("display.max_colwidth",None)

DATA_DIR = Path("../data/raw/")
meta_df = pd.read_json(DATA_DIR/"meta_Video_Games.jsonl",lines=True)
splits = {
    name : pd.read_csv(DATA_DIR/ f"Video_Games.{name}.csv") for name in ("train","valid","test")
}
train,valid,test = splits["train"],splits["valid"],splits["test"]
all_items = pd.concat([train[["parent_asin"]],valid[["parent_asin"]],test[["parent_asin"]]],ignore_index=True).drop_duplicates()
display(meta_df.head(2))
items_df = all_items.merge(meta_df,on="parent_asin",how="inner",validate="one_to_one")
display(items_df.head(2))
print(items_df.shape)
assert len(items_df) == len(all_items), "some items have no metadata row"
print(f"coverage: all {len(all_items):,} items have a metadata row")


,main_category,title,average_rating,rating_number,features,description,price,images,videos,store,categories,details,parent_asin,bought_together,subtitle,author
0,Video Games,Dash 8-300 Professional Add-On,5.0,1,[Features Dash 8-300 and 8-Q300 ('Q' rollout l...,[The Dash 8-300 Professional Add-On lets you p...,None,[{'thumb': 'https://m.media-amazon.com/images/...,[],Aerosoft,"[Video Games, PC, Games]",{'Pricing': 'The strikethrough price is the Li...,B000FH0MHO,NaN,NaN,NaN
1,Video Games,Phantasmagoria: A Puzzle of Flesh,4.1,18,[Windows 95],[],None,[{'thumb': 'https://m.media-amazon.com/images/...,[],Sierra,"[Video Games, PC, Games]","{'Best Sellers Rank': {'Video Games': 137612, ...",B00069EVOG,NaN,NaN,NaN


,parent_asin,main_category,title,average_rating,rating_number,features,description,price,images,videos,store,categories,details,bought_together,subtitle,author
0,B08R5B7YS4,All Electronics,Advanced 3D Deluxe Audio Strap Adapter Kit for...,4.0,88,[The best for Oculus Quest 2 DAS adapter now u...,[Does NOT include Oculus Quest 2 Headset or VI...,None,[{'thumb': 'https://m.media-amazon.com/images/...,[],Advanced 3D,"[Video Games, PC, Virtual Reality, Accessories]",{'Package Dimensions': '5.83 x 4.57 x 0.47 inc...,NaN,NaN,NaN
1,B0863MT183,All Electronics,KIWI design VR Stand Compatible with Quest 2/R...,4.5,1638,[Stable and Safe - KIWI design VR stand is spe...,[],19.99,[{'thumb': 'https://m.media-amazon.com/images/...,"[{'title': 'Makes oculus look nice', 'url': 'h...",KIWI design,"[Video Games, PC, Virtual Reality, Headsets]",{'Product Dimensions': '10.63 x 6.69 x 7.09 in...,NaN,NaN,NaN


(25612, 16)
coverage: all 25,612 items have a metadata row


## 2. How complete is each field?
Missing = None/NaN, blank string, or empty list/dict. Compared for all items and for the 1,542 items unseen in train+valid.

In [2]:
# items that appear in test but never in train or valid
seen_ids = set(train["parent_asin"]) | set(valid["parent_asin"])
unseen_ids = set(test["parent_asin"]) - seen_ids
items_df["is_unseen"] = items_df["parent_asin"].isin(unseen_ids)
print("unseen items:",items_df["is_unseen"].sum())

def is_empty(x):
     """True for None, NaN, blank strings, and empty lists/dicts."""
     if x is None:
          return True
     if isinstance(x,float) and pd.isna(x):
          return True
     if isinstance(x,str):
          return x.strip() == ""
     if isinstance(x,(list,dict)):
          return len(x) == 0
     return False

fields = [c for c in items_df.columns if c not in ("parent_asin","is_unseen")]
empty_flags  = items_df[fields].apply(lambda col : col.map(is_empty))

report = pd.DataFrame({
     "all_items_%" : empty_flags.mean()*100,
     "unseen_%" : empty_flags[items_df["is_unseen"]].mean()*100,
}).round(1)

report.sort_values("all_items_%",ascending=False)


unseen items: 1542


,all_items_%,unseen_%
bought_together,100.0,100.0
subtitle,100.0,100.0
author,100.0,100.0
videos,58.6,20.9
price,33.1,15.1
description,26.6,78.0
features,12.9,2.9
categories,3.4,1.5
details,2.2,1.1
main_category,2.0,3.4


## 3. What is in the catalog?
Question: what share of items are real games versus accessories or hardware, and does the unseen group look different?

In [3]:
def side_by_side(series,top=10):
    """Share of items (%) per value, for all items and for the unseen ones."""
    out = pd.DataFrame({
        "all_items_%" : series.value_counts(normalize = True)*100,
        "unseen_%" : series[items_df["is_unseen"]].value_counts(normalize = True) * 100,
    }).round(1)
    return out.sort_values("all_items_%",ascending=False).head(top)

main_cat = items_df["main_category"].fillna("(missing)")
cats = items_df["categories"].map(lambda c: c if isinstance(c, list) else [])
platform = cats.map(lambda c: c[1] if len(c) > 1 else "(none)")   # 2nd entry: a platform (PC, PlayStation 4) or "Legacy Systems" for old consoles
level3 = cats.map(lambda c: c[2] if len(c) > 2 else "(none)")     # 3rd entry: Games/Accessories/VR for modern platforms, console family for legacy
accessory = cats.map(lambda c: "accessory" if "Accessories" in c else "other")

for title, s in [("main_category", main_cat), ("2nd category entry", platform),
                 ("3rd category entry", level3), ("accessory?", accessory)]:
    print(f"--- {title} ---")
    print(side_by_side(s), "\n")

--- main_category ---
                           all_items_%  unseen_%
main_category                                   
Video Games                       70.0      26.8
Computers                         12.7      15.7
All Electronics                   10.3      42.9
(missing)                          2.0       3.4
Cell Phones & Accessories          1.6       6.4
Toys & Games                       0.8       0.9
Software                           0.8       0.1
Industrial & Scientific            0.5       2.1
Amazon Home                        0.2       0.5
Tools & Home Improvement           0.2       0.4 

--- 2nd category entry ---
                             all_items_%  unseen_%
categories                                        
Legacy Systems                      45.7      17.3
PC                                  24.8      38.7
Nintendo Switch                      8.9      22.2
PlayStation 4                        7.1       5.0
Xbox One                             5.6       4.0
(non

## Findings
- Every one of the 25,612 items has exactly one metadata row (including the 1,542 items unseen in train+valid), so coverage is complete.
- `bought_together`, `subtitle` and `author` are empty for every item and will be ignored.
- Very complete: `title` (1 missing), `store` (0.5% empty), `categories` (3.4%), `main_category` (2.0%).
- `price` is empty for 33% of items (15% of unseen items): use a "price missing" flag.
- `description` is empty for 78% of the unseen items (27% overall), while `features` (bullet points) is filled for 97% of them: use `title` + `features` as the item text, not `description`.
- `average_rating` and `rating_number` are filled for every item but were recorded at crawl time and include future reviews, so they must be recomputed from the training period.
- The catalog is not only games: 40% of items have "Accessories" in their categories, and 70% of the unseen items do.
- Unseen items skew towards newer hardware: Nintendo Switch (22% vs 9%), PlayStation 5 (6% vs 1%), Xbox Series X|S (3% vs 0.5%) and Virtual Reality (16% vs 2%). By `main_category`, only 27% of unseen items are "Video Games" (70% overall); 43% are "All Electronics".
- `categories` is a hierarchy whose levels are not consistent. For modern platforms the third entry in the list is the type (Games/Accessories/VR), but for "Legacy Systems" (the most common second entry: 46% of items, 17% of unseen items) it is the console family. Fixed positions are unreliable, so the item tower should use all category names as tokens.

## Next steps
1. Choose the item-tower features: title + feature bullets, all category names as tokens, store, price with a missing flag.
2. Define the evaluation setup (per-row vs per-user, rows with no history, seen vs new items).
3. Build the baselines (most popular, co-occurrence).